In [13]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
import numpy as np
from tqdm import tqdm

# ============================================================================
# CONFIGURATION
# ============================================================================

CHARACTERS = [
    'Intervention', 'Barrier', 'CrossingSignal',
    'Man', 'Woman', 'Pregnant', 'Stroller', 'OldMan', 'OldWoman',
    'Boy', 'Girl', 'Homeless', 'LargeWoman', 'LargeMan', 'Criminal',
    'MaleExecutive', 'FemaleExecutive', 'FemaleAthlete', 'MaleAthlete',
    'FemaleDoctor', 'MaleDoctor', 'Dog', 'Cat'
]

NUM_CHARACTERS = len(CHARACTERS)
MAX_CARDINALITY = 10
NUM_TEAMS = 2
EMBED_DIM = 128
NUM_HEADS = 8
NUM_LAYERS = 6
DROPOUT = 0.05


# ============================================================================
# DATASET
# ============================================================================

class MoralChoiceDataset(Dataset):
    def __init__(self, scenarios, labels):
        """
        Args:
            scenarios: (N, 2, 23) array
            labels: (N, 1) array
        """
        # Store as numpy for memory efficiency
        self.scenarios = scenarios
        self.labels = labels

    def __len__(self):
        return len(self.scenarios)

    def __getitem__(self, idx):
        # Convert to tensor only when accessed
        scenario = torch.tensor(self.scenarios[idx], dtype=torch.long)
        label = torch.tensor(self.labels[idx], dtype=torch.float32)
        return scenario, label


# ============================================================================
# MODEL
# ============================================================================

class MoralReasoningTransformer(nn.Module):
    def __init__(
        self,
        num_characters=NUM_CHARACTERS,
        max_cardinality=MAX_CARDINALITY,
        num_teams=NUM_TEAMS,
        embed_dim=EMBED_DIM,
        num_heads=NUM_HEADS,
        num_layers=NUM_LAYERS,
        dropout=DROPOUT
    ):
        super().__init__()

        self.num_characters = num_characters
        self.embed_dim = embed_dim

        # Compositional embeddings
        self.character_embedding = nn.Embedding(num_characters, embed_dim)
        self.cardinality_embedding = nn.Embedding(max_cardinality + 1, embed_dim)
        self.team_embedding = nn.Embedding(num_teams, embed_dim)

        # Transformer encoder
        encoder_layer = nn.TransformerEncoderLayer(
            d_model=embed_dim,
            nhead=num_heads,
            dim_feedforward=embed_dim * 4,
            dropout=dropout,
            batch_first=True,
            norm_first=True
        )
        self.transformer = nn.TransformerEncoder(encoder_layer, num_layers=num_layers)

        # CLS token for aggregation
        self.cls_token = nn.Parameter(torch.randn(1, 1, embed_dim))

        # Classification head on CLS token
        self.classifier = nn.Sequential(
            nn.LayerNorm(embed_dim),
            nn.Linear(embed_dim, embed_dim // 2),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(embed_dim // 2, 1)
        )

    def encode_outcome(self, counts, team_id):
        batch_size = counts.shape[0]

        character_ids = torch.arange(
            self.num_characters,
            device=counts.device
        ).unsqueeze(0).expand(batch_size, -1)

        char_emb = self.character_embedding(character_ids)
        card_emb = self.cardinality_embedding(counts)

        team_id_tensor = torch.full(
            (batch_size, self.num_characters),
            team_id,
            device=counts.device,
            dtype=torch.long
        )
        team_emb = self.team_embedding(team_id_tensor)

        tokens = char_emb + card_emb + team_emb

        return tokens

    def forward(self, scenarios):
        batch_size = scenarios.shape[0]

        outcome_0 = scenarios[:, 0, :]
        outcome_1 = scenarios[:, 1, :]

        tokens_0 = self.encode_outcome(outcome_0, team_id=0)
        tokens_1 = self.encode_outcome(outcome_1, team_id=1)

        cls_tokens = self.cls_token.expand(batch_size, -1, -1)
        all_tokens = torch.cat([cls_tokens, tokens_0, tokens_1], dim=1)

        encoded = self.transformer(all_tokens)
        cls_output = encoded[:, 0, :]
        logits = self.classifier(cls_output)

        return logits


# ============================================================================
# TRAINING
# ============================================================================

def train_epoch(model, dataloader, optimizer, scheduler, device, epoch, grad_clip=1.0):
    model.train()
    total_loss = 0
    correct = 0
    total = 0

    pbar = tqdm(dataloader, desc=f"Epoch {epoch} [Train]")

    for batch_idx, (scenarios, labels) in enumerate(pbar):
        scenarios = scenarios.to(device)
        labels = labels.to(device)

        optimizer.zero_grad()

        logits = model(scenarios)
        loss = F.binary_cross_entropy_with_logits(
            logits.squeeze(-1),
            labels.squeeze(-1)
        )

        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), grad_clip)
        optimizer.step()

        total_loss += loss.item()
        preds = (torch.sigmoid(logits.squeeze(-1)) > 0.5).float()
        correct += (preds == labels.squeeze(-1)).sum().item()
        total += len(labels)

        # Update progress bar every 10 batches
        if batch_idx % 10 == 0:
            current_acc = correct / total if total > 0 else 0
            pbar.set_postfix({
                'loss': f'{loss.item():.4f}',
                'acc': f'{current_acc:.4f}'
            })

        scheduler.step()

    return total_loss / len(dataloader), correct / total


def evaluate(model, dataloader, device, epoch):
    model.eval()
    total_loss = 0
    correct = 0
    total = 0

    pbar = tqdm(dataloader, desc=f"Epoch {epoch} [Val]")

    with torch.no_grad():
        for scenarios, labels in pbar:
            scenarios = scenarios.to(device)
            labels = labels.to(device)

            logits = model(scenarios)
            loss = F.binary_cross_entropy_with_logits(
                logits.squeeze(-1),
                labels.squeeze(-1)
            )

            total_loss += loss.item()
            preds = (torch.sigmoid(logits.squeeze(-1)) > 0.5).float()
            correct += (preds == labels.squeeze(-1)).sum().item()
            total += len(labels)

    return total_loss / len(dataloader), correct / total


# ============================================================================
# MAIN
# ============================================================================

def train_model(
    data,
    batch_size=32,
    learning_rate=1e-4,
    num_epochs=10,
    weight_decay=0.01,
    warmup_steps=1000,
    save_path='best_model.pt',
    val_split=0.1
):
    print("="*80)
    print("MORAL REASONING TRANSFORMER - TRAINING")
    print("="*80)

    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    print(f"\n[DEVICE] Training on {device}")

    if torch.cuda.is_available():
        print(f"[GPU] {torch.cuda.get_device_name(0)}")
        print(f"[GPU] Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.2f} GB")

    scenarios, labels = data
    print(f"\n[DATA] Total samples: {len(scenarios):,}")
    print(f"[DATA] Scenarios shape: {scenarios.shape}")
    print(f"[DATA] Labels shape: {labels.shape}")
    print(f"[DATA] Data type: {scenarios.dtype}")

    # Convert to numpy if needed (memory efficient)
    if isinstance(scenarios, torch.Tensor):
        print("[DATA] Converting scenarios from tensor to numpy...")
        scenarios = scenarios.numpy()
    if isinstance(labels, torch.Tensor):
        print("[DATA] Converting labels from tensor to numpy...")
        labels = labels.numpy()

    # Split train/val
    print(f"\n[SPLIT] Creating {val_split*100:.0f}% validation split...")
    n_total = len(scenarios)
    n_val = int(n_total * val_split)
    n_train = n_total - n_val

    print(f"[SPLIT] Shuffling indices...")
    indices = np.random.permutation(n_total)
    train_idx = indices[:n_train]
    val_idx = indices[n_train:]

    print(f"[SPLIT] Train samples: {n_train:,}")
    print(f"[SPLIT] Val samples: {n_val:,}")

    # Create datasets (keep as numpy, convert to tensor in __getitem__)
    print("\n[DATASET] Creating train dataset...")
    train_dataset = MoralChoiceDataset(scenarios[train_idx], labels[train_idx])
    print("[DATASET] Creating val dataset...")
    val_dataset = MoralChoiceDataset(scenarios[val_idx], labels[val_idx])

    print(f"\n[DATALOADER] Creating train dataloader (batch_size={batch_size})...")
    train_loader = DataLoader(
        train_dataset,
        batch_size=batch_size,
        shuffle=True,
        num_workers=0,  # Reduced for memory
        pin_memory=True,
    )

    print(f"[DATALOADER] Creating val dataloader (batch_size={batch_size})...")
    val_loader = DataLoader(
        val_dataset,
        batch_size=batch_size,
        num_workers=0,
        pin_memory=True,
    )

    print(f"[DATALOADER] Train batches: {len(train_loader):,}")
    print(f"[DATALOADER] Val batches: {len(val_loader):,}")

    # Initialize model
    print(f"\n[MODEL] Initializing model...")
    model = MoralReasoningTransformer().to(device)

    num_params = sum(p.numel() for p in model.parameters())
    print(f"[MODEL] Total parameters: {num_params:,}")
    print(f"[MODEL] Embedding dim: {EMBED_DIM}")
    print(f"[MODEL] Num heads: {NUM_HEADS}")
    print(f"[MODEL] Num layers: {NUM_LAYERS}")

    print(f"\n[OPTIMIZER] AdamW (lr={learning_rate}, weight_decay={weight_decay})")
    optimizer = torch.optim.AdamW(
        model.parameters(),
        lr=learning_rate,
        weight_decay=weight_decay,
        betas=(0.9, 0.999)
    )

    # Scheduler
    total_steps = len(train_loader) * num_epochs
    print(f"[SCHEDULER] OneCycleLR (total_steps={total_steps:,}, warmup={warmup_steps})")
    scheduler = torch.optim.lr_scheduler.OneCycleLR(
        optimizer,
        max_lr=learning_rate,
        total_steps=total_steps,
        pct_start=warmup_steps / total_steps,
        anneal_strategy='cos'
    )

    print("\n" + "="*80)
    print("STARTING TRAINING")
    print("="*80 + "\n")

    best_val_acc = 0
    patience = 10
    patience_counter = 0

    for epoch in range(1, num_epochs + 1):
        print(f"\n{'='*80}")
        print(f"EPOCH {epoch}/{num_epochs}")
        print(f"{'='*80}")

        train_loss, train_acc = train_epoch(
            model, train_loader, optimizer, scheduler, device, epoch
        )
        val_loss, val_acc = evaluate(model, val_loader, device, epoch)

        current_lr = optimizer.param_groups[0]['lr']

        print(f"\n[RESULTS] Epoch {epoch:03d}/{num_epochs}")
        print(f"  Learning Rate: {current_lr:.2e}")
        print(f"  Train: Loss={train_loss:.4f} | Acc={train_acc:.4f}")
        print(f"  Val:   Loss={val_loss:.4f}   | Acc={val_acc:.4f}")

        if val_acc > best_val_acc:
            best_val_acc = val_acc
            patience_counter = 0
            print(f"  ✓ NEW BEST! Saving model to {save_path}")
            torch.save({
                'epoch': epoch,
                'model_state_dict': model.state_dict(),
                'optimizer_state_dict': optimizer.state_dict(),
                'val_acc': val_acc,
                'val_loss': val_loss,
                'train_acc': train_acc,
                'train_loss': train_loss
            }, save_path)
        else:
            patience_counter += 1
            print(f"  No improvement ({patience_counter}/{patience})")
            if patience_counter >= patience:
                print(f"\n[EARLY STOP] No improvement for {patience} epochs")
                break

    print("\n" + "="*80)
    print("TRAINING COMPLETE")
    print(f"Best validation accuracy: {best_val_acc:.4f}")
    print("="*80 + "\n")

    return model


In [14]:
import pickle
with open("/kaggle/input/moral-vectors/moral_machine_vector_data.pkl", "rb") as f:
    data = pickle.load(f)


In [15]:
data = (np.array(data[0]), np.array(data[1]))

In [ ]:
model = train_model(
    data,
    batch_size=512,
    learning_rate=1e-4,
    num_epochs=10
)

MORAL REASONING TRANSFORMER - TRAINING

[DEVICE] Training on cuda
[GPU] Tesla P100-PCIE-16GB
[GPU] Memory: 17.06 GB

[DATA] Total samples: 5,449,620
[DATA] Scenarios shape: (5449620, 2, 23)
[DATA] Labels shape: (5449620,)
[DATA] Data type: int64

[SPLIT] Creating 10% validation split...
[SPLIT] Shuffling indices...
[SPLIT] Train samples: 4,904,658
[SPLIT] Val samples: 544,962

[DATASET] Creating train dataset...
[DATASET] Creating val dataset...

[DATALOADER] Creating train dataloader (batch_size=512)...
[DATALOADER] Creating val dataloader (batch_size=512)...
[DATALOADER] Train batches: 9,580
[DATALOADER] Val batches: 1,065

[MODEL] Initializing model...
[MODEL] Total parameters: 1,202,945
[MODEL] Embedding dim: 128
[MODEL] Num heads: 8
[MODEL] Num layers: 6

[OPTIMIZER] AdamW (lr=0.0001, weight_decay=0.01)
[SCHEDULER] OneCycleLR (total_steps=95,800, warmup=1000)

STARTING TRAINING


EPOCH 1/10


Epoch 1 [Val]: 100%|██████████| 1065/1065 [00:31<00:00, 34.09it/s]



[RESULTS] Epoch 001/10
  Learning Rate: 9.80e-05
  Train: Loss=0.5358 | Acc=0.7357
  Val:   Loss=0.5239   | Acc=0.7463
  ✓ NEW BEST! Saving model to best_model.pt

EPOCH 2/10


Epoch 2 [Val]: 100%|██████████| 1065/1065 [00:31<00:00, 33.67it/s]



[RESULTS] Epoch 002/10
  Learning Rate: 9.12e-05
  Train: Loss=0.5236 | Acc=0.7465
  Val:   Loss=0.5237   | Acc=0.7470
  ✓ NEW BEST! Saving model to best_model.pt

EPOCH 3/10


Epoch 3 [Train]:  66%|██████▌   | 6289/9580 [10:15<05:20, 10.27it/s, loss=0.5675, acc=0.7472]